1. Master Setup and Data Loading

In [13]:
# --- MASTER IMPORTS: Data Handling, Visualization, and Modeling ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from google.colab import drive # For accessing Google Drive in Colab

# --- STEP 1: Connect to Google Drive ---
# This step prompts you to authorize Google Colab to access files on your Drive.
drive.mount('/content/drive')

# --- STEP 2: Define File Paths (CRITICAL: Adjust this path!) ---
# You must replace 'YOUR_FOLDER_PATH' with the actual path to your Titanic files on Drive.
# Example: '/content/drive/MyDrive/DataScience_Internship/train.csv'
file_path_train = '/content/drive/MyDrive/titanic/train.csv'
file_path_test = '/content/drive/MyDrive/titanic/test.csv'

# --- STEP 3: Load Datasets ---
train_data = pd.read_csv(file_path_train)
test_data = pd.read_csv(file_path_test)

# --- STEP 4: Prepare Data for Consistent Preprocessing ---
# Store the Passenger IDs from the test set for the final submission.
test_passenger_ids = test_data['PassengerId']

# Combine the datasets. We drop 'Survived' from the training set before joining
# because the test set does not have this column (it's what we need to predict).
combined_data = pd.concat([train_data.drop('Survived', axis=1), test_data],
                         ignore_index=True, sort=False)

# Display a quick check to confirm successful loading
print(f"Training Data shape: {train_data.shape}")
print(f"Test Data shape: {test_data.shape}")
print(f"Combined Data shape: {combined_data.shape}")
print("Cell 1: Setup and Data Loading Complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training Data shape: (891, 12)
Test Data shape: (418, 11)
Combined Data shape: (1309, 11)
Cell 1: Setup and Data Loading Complete.


Inspecting Data, Data Types and Missing Values

In [14]:
# --- STEP 1: View the first 5 rows of the Training Data ---
# This gives a quick visual sense of the data's content, format, and structure.
print("--- Training Data Snapshot (First 5 Rows) ---")
print(train_data.head())

# --- STEP 2: Display all Column Names ---
# This confirms the exact spelling of features we need to reference later.
print("\n" + "="*50 + "\n")
print("--- All Columns in the Training Data ---")
print(train_data.columns.tolist())

# --- STEP 3: Check the end of the Test Data ---
# It's good practice to look at the test data as well, especially the end,
# to check for any weird formatting or missing values specific to that file.
print("\n" + "="*50 + "\n")
print("--- Test Data Snapshot (Last 5 Rows) ---")
print(test_data.tail())

--- Training Data Snapshot (First 5 Rows) ---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0 

In [15]:
# --- STEP 1: Data Structure Overview and Non-Null Counts ---
# Check data types and quickly identify which columns contain missing values.
print("--- 1. Data Structure and Non-Null Counts ---")
combined_data.info()

# --- STEP 2: Statistical Summary of Numerical Data ---
# View the central tendency, spread, and range of numerical features.
print("\n" + "="*50 + "\n")
print("--- 2. Descriptive Statistics for Numerical Features ---")
print(combined_data.describe())

# --- STEP 3: Quantify Missing Data for Imputation Strategy ---
# Determine the exact scale of the missing data problem.
print("\n" + "="*50 + "\n")
print("--- 3. Missing Data Summary ---")

# Calculate the number of missing values for every column.
missing_totals = combined_data.isnull().sum()

# Calculate the percentage of missing values relative to the total number of rows.
missing_percentages = (missing_totals / len(combined_data)) * 100

# Create a consolidated DataFrame containing only the columns with missing data.
missing_data_report = pd.DataFrame({
    'Total Missing': missing_totals,
    'Percent Missing': missing_percentages
})

# Filter the report to show only columns where Total Missing > 0, and sort by percentage.
columns_to_fix = missing_data_report[missing_data_report['Total Missing'] > 0].sort_values(
    by='Percent Missing', ascending=False
)

print(columns_to_fix)
print("\nDiagnosis Complete. We have a clear plan for data cleaning!")

--- 1. Data Structure and Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Pclass       1309 non-null   int64  
 2   Name         1309 non-null   object 
 3   Sex          1309 non-null   object 
 4   Age          1046 non-null   float64
 5   SibSp        1309 non-null   int64  
 6   Parch        1309 non-null   int64  
 7   Ticket       1309 non-null   object 
 8   Fare         1308 non-null   float64
 9   Cabin        295 non-null    object 
 10  Embarked     1307 non-null   object 
dtypes: float64(2), int64(4), object(5)
memory usage: 112.6+ KB


--- 2. Descriptive Statistics for Numerical Features ---
       PassengerId       Pclass          Age        SibSp        Parch  \
count  1309.000000  1309.000000  1046.000000  1309.000000  1309.000000   
mean    655.000000     2.29488

Creating New Predictive Features


In [16]:
# --- STEP 1: Feature Engineering - Title Extraction from Name

# We use an 'r' before the string to denote a 'raw string' (r' pattern '), which is best practice for regex.
combined_data['Title'] = combined_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Group rare titles into a single 'Rare' category to prevent overfitting
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
combined_data['Title'] = combined_data['Title'].replace(rare_titles, 'Rare')

# Normalize the common titles for better consistency
combined_data['Title'] = combined_data['Title'].replace(['Mlle', 'Ms', 'Mme'], ['Miss', 'Miss', 'Mrs'])


# --- STEP 2: Feature Engineering - Family Size and Alone Status ---

# 'SibSp' is siblings/spouses. 'Parch' is parents/children. Add 1 for the passenger themselves.
combined_data['FamilySize'] = combined_data['SibSp'] + combined_data['Parch'] + 1

# Create a binary feature: 1 if the passenger traveled alone, 0 otherwise.
# np.where is a clear and fast way to create conditional columns.
combined_data['IsAlone'] = np.where(combined_data['FamilySize'] == 1, 1, 0)


# --- STEP 3: Feature Engineering - Deck Extraction from Cabin ---

# The robust way to fill NaNs: assign the result back to the column.
combined_data['Cabin'] = combined_data['Cabin'].fillna('U')

# Extract the first letter of the Cabin, which represents the Deck level.
combined_data['Deck'] = combined_data['Cabin'].str[0]


# --- STEP 4: Inspect the new features ---
print("--- New Feature Distribution (Title) ---")
print(combined_data['Title'].value_counts())

print("\n--- New Feature Distribution (Deck) ---")
print(combined_data['Deck'].value_counts())
print("\nFeature Engineering Complete. Data is now richer and warnings are resolved!")

--- New Feature Distribution (Title) ---
Title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64

--- New Feature Distribution (Deck) ---
Deck
U    1014
C      94
B      65
D      46
E      41
A      22
F      21
G       5
T       1
Name: count, dtype: int64

Feature Engineering Complete. Data is now richer and warnings are resolved!


Handling Missing Values and Categorical Encoding

In [17]:
# --- STEP 1: Impute Missing Values ---

# 1.1 Impute the single missing 'Fare' value with the median.
median_fare = combined_data['Fare'].median()
combined_data['Fare'] = combined_data['Fare'].fillna(median_fare)

# 1.2 Impute the two missing 'Embarked' values with the mode.
mode_embarked = combined_data['Embarked'].mode()[0]
combined_data['Embarked'] = combined_data['Embarked'].fillna(mode_embarked)

# 1.3 Smart Imputation for 'Age' using the median of each 'Title' group.
# We use .transform() to apply the median of each group back to the original column's NaNs.
combined_data['Age'] = combined_data.groupby('Title')['Age'].transform(
    lambda x: x.fillna(x.median())
)

# --- STEP 2: Drop Unnecessary and Processed Features ---
# These columns are either text-based, too sparse, or have been successfully engineered.
columns_to_drop = ['Name', 'Ticket', 'Cabin', 'SibSp', 'Parch']
# We do not drop 'PassengerId' yet, as we need to split the data first!
combined_data.drop(columns=columns_to_drop, axis=1, inplace=True)


# --- STEP 3: Categorical Encoding (One-Hot Encoding) ---
# Identify all columns that are categorical and need to be converted to numbers (0s and 1s).
# Pclass is included here because it's better treated as a category (1, 2, 3) than an ordered number.
categorical_cols = ['Sex', 'Embarked', 'Deck', 'Title', 'Pclass', 'IsAlone']
combined_data = pd.get_dummies(combined_data, columns=categorical_cols, drop_first=False)

# --- STEP 4: Final Check ---
print("--- Final Check on Data Structure ---")
combined_data.info()
print("\nData Preprocessing Complete. The data is clean and numerical!")

--- Final Check on Data Structure ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   1309 non-null   int64  
 1   Age           1309 non-null   float64
 2   Fare          1309 non-null   float64
 3   FamilySize    1309 non-null   int64  
 4   Sex_female    1309 non-null   bool   
 5   Sex_male      1309 non-null   bool   
 6   Embarked_C    1309 non-null   bool   
 7   Embarked_Q    1309 non-null   bool   
 8   Embarked_S    1309 non-null   bool   
 9   Deck_A        1309 non-null   bool   
 10  Deck_B        1309 non-null   bool   
 11  Deck_C        1309 non-null   bool   
 12  Deck_D        1309 non-null   bool   
 13  Deck_E        1309 non-null   bool   
 14  Deck_F        1309 non-null   bool   
 15  Deck_G        1309 non-null   bool   
 16  Deck_T        1309 non-null   bool   
 17  Deck_U        1309 non-null   boo

In [18]:
combined_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   1309 non-null   int64  
 1   Age           1309 non-null   float64
 2   Fare          1309 non-null   float64
 3   FamilySize    1309 non-null   int64  
 4   Sex_female    1309 non-null   bool   
 5   Sex_male      1309 non-null   bool   
 6   Embarked_C    1309 non-null   bool   
 7   Embarked_Q    1309 non-null   bool   
 8   Embarked_S    1309 non-null   bool   
 9   Deck_A        1309 non-null   bool   
 10  Deck_B        1309 non-null   bool   
 11  Deck_C        1309 non-null   bool   
 12  Deck_D        1309 non-null   bool   
 13  Deck_E        1309 non-null   bool   
 14  Deck_F        1309 non-null   bool   
 15  Deck_G        1309 non-null   bool   
 16  Deck_T        1309 non-null   bool   
 17  Deck_U        1309 non-null   bool   
 18  Title_Master  1309 non-null 

Data Splitting and Model Selection

In [19]:
# --- STEP 1: Separate the Cleaned Data Back into Train and Test Sets ---

# We split the cleaned combined_data back into its original train and test parts
# using the length of the original training data (891 rows) as the split point.
# This ensures the features are aligned with the original target variable (y).
X_train = combined_data.iloc[:len(train_data), :]
X_test = combined_data.iloc[len(train_data):, :]

# Define the target variable (y) from the original loaded training data
y_train = train_data['Survived']

# Drop the 'PassengerId' from the features as it's an arbitrary ID and adds no predictive value.
X_train = X_train.drop('PassengerId', axis=1)
X_test = X_test.drop('PassengerId', axis=1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")


# --- STEP 2: Initialize and Train the Model (Random Forest) ---

# We choose the Random Forest Classifier, a powerful, robust ensemble model.
# n_estimators=100 means the forest will have 100 individual decision trees.
# random_state=42 ensures that the results are the same every time we run the code (reproducibility).
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=8, min_samples_leaf=3)

# The fit method is where the model learns the relationship between the features (X_train)
# and the target (y_train).
model.fit(X_train, y_train)

print("\n Model Training Complete. The Random Forest model has learned the patterns!")

X_train shape: (891, 27)
X_test shape: (418, 27)
y_train shape: (891,)

 Model Training Complete. The Random Forest model has learned the patterns!


Making Predictions on the Test Set

In [20]:
# --- STEP 1: Evaluate Training Accuracy ---
# Predict on the training data (X_train) to get a baseline performance metric.
y_train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy Score: {train_accuracy:.4f}")


# --- STEP 2: Generate Predictions for the Test Set ---
# This is the core prediction step on the unseen data (X_test).
predictions = model.predict(X_test)

# --- STEP 3: Display the first few predictions ---
print("\nFirst 10 predictions on the Test Set (0=Died, 1=Survived):")
print(predictions[:10])

print("\nPredictions Generated. Ready for submission formatting.")

Training Accuracy Score: 0.8709

First 10 predictions on the Test Set (0=Died, 1=Survived):
[0 0 0 0 1 0 1 0 1 0]

Predictions Generated. Ready for submission formatting.


Submission File Generation

In [21]:
# --- STEP 1: Create the Submission DataFrame ---

# Match the Passenger IDs (saved from the original test file) with the model's predictions.
submission_df = pd.DataFrame({
    # Use the IDs saved from the start of the project
    'PassengerId': test_passenger_ids,
    # Use the model's output (the 0s and 1s)
    'Survived': predictions
})

# Cast the 'Survived' column to the required integer format (0 or 1).
submission_df['Survived'] = submission_df['Survived'].astype(int)


# --- STEP 2: Export the File ---

# Export the DataFrame to a CSV file. This file is the final deliverable
# VITAL: index=False ensures no unnecessary row index column is included.
submission_df.to_csv('titanic_submission_rf_v1.csv', index=False)

print("--- Submission Deliverable File Head (First 5 Rows) ---")
print(submission_df.head())
print("\nFinal Prediction Deliverable 'titanic_submission_rf_v1.csv' created successfully.")

--- Submission Deliverable File Head (First 5 Rows) ---
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1

Final Prediction Deliverable 'titanic_submission_rf_v1.csv' created successfully.
